# 🧠 SVOMPTR-9B: Master Neural Distillation & MoE Training

This notebook covers:
- 🛡️ Google Drive Integration (`svomptr_brain`)
- 🗃️ Massive 5M Data Generation via vLLM
- 🧺 Fast Knowledge Distillation & DOP Alignment
- 🔄 Auto-resume & Idle Prevention
- 🚀 LocalTunnel ML API Deployment

### 🛡️ 1. Prevent Idle Timeout
Press `Ctrl+Shift+I` (or `Cmd+Option+I` on Mac) to open browser Developer Tools. Go to the **Console** tab, paste the following code, and press Enter. This will keep Colab alive during long training sessions.

```javascript
function KeepClicking(){
  console.log("Keeping Colab active...");
  document.querySelector("colab-toolbar-button").click();
}
setInterval(KeepClicking, 60000);
```

In [ ]:
# 🛡️ 2. Drive Mount & Brain Initialization
from google.colab import drive
import os

drive.mount('/content/drive')
brain_dir = '/content/drive/MyDrive/svomptr_brain'

# Create necessary directories
directories = ['datasets', 'checkpoints', 'weights', 'dop_alignment']
for d in directories:
    os.makedirs(os.path.join(brain_dir, d), exist_ok=True)

print(f"✅ SVOMPTR Brain active at: {brain_dir}")

In [ ]:
# ⚙️ 3. Install Core Dependencies
print("Installing core dependencies... This may take a few minutes.")
!pip install -q --upgrade pip transformers accelerate bitsandbytes datasets sentence-transformers fastapi uvicorn pydantic pyngrok vllm trl

import torch
if torch.cuda.is_available():
    print("✅ GPU Detected. Fine-tuning and high-speed generation enabled.")
    !pip install faiss-gpu
else:
    print("⚠️ CPU Only Mode. Installing CPU-optimized FAISS and skipping vLLM.")
    !pip install faiss-cpu


In [ ]:
# 📂 4. Project Setup
import sys
import os
repo_path = '/content/svomptr-project'
svo_9b_path = os.path.join(repo_path, 'svomptr_9b')
if not os.path.exists(repo_path):
    print("Cloning SVOMPTR repository to /content/svomptr-project...")
    !git clone https://github.com/kkomyoeminaung/svomptr-9b-moe-model-.git /content/svomptr-project
sys.path.insert(0, repo_path)
sys.path.insert(0, svo_9b_path)

### 🗃️ 5. Massive Synthetic Data Generation (vLLM for Speed)
Generates millions of structured SVOMPTR English-Myanmar pairs rapidly. Resumes automatically if interrupted.

In [ ]:
%%writefile generate_data.py
import os
import sys
import torch

# Ensure project library is in path
repo_path = '/content/svomptr-project'
svo_9b_path = os.path.join(repo_path, 'svomptr_9b')
for p in [repo_path, svo_9b_path]:
    if p not in sys.path: sys.path.insert(0, p)

from svomptr.distillation.pipeline import run_distillation_pipeline

def main():
    print("🚀 Starting SVOMPTR High-Speed Generation Pipeline (5M Target)...")
    BRAIN_DIR = os.environ.get('SVOMPTR_BRAIN_PATH', '/content/drive/MyDrive/svomptr_brain')
    DATASET_FILE = os.path.join(BRAIN_DIR, 'datasets', 'synthetic_5000000.jsonl')
    
    # Auto-detect hardware for optimal generation speed
    use_vllm = torch.cuda.is_available()
    
    # Execute robust pipeline with resume support and heartbeat logs
    run_distillation_pipeline(
        output_file=DATASET_FILE,
        dry_run=False,
        use_vllm=use_vllm,
        target_total_samples=5000000
    )

if __name__ == '__main__':
    main()


In [ ]:
!python generate_data.py

### 🧺 6. Distillation & DOP Alignment Training
Train the Chat Expert student model using the synthetic dataset. Auto-saves to Drive and resumes on interruption.

In [ ]:
%%writefile train_distillation.py
import os
import sys

# Ensure project library is in path
repo_path = '/content/svomptr-project'
if repo_path not in sys.path: sys.path.insert(0, repo_path)

from svomptr_moe.train_chat_expert import train_chat_expert

def main():
    print("🧠 Starting SVOMPTR Knowledge Distillation Training...")
    # The training script handles everything: 
    # - Loading 5M dataset from Drive
    # - Checkpoint resume logic
    # - Saving final weights and DOP signature
    train_chat_expert()

if __name__ == '__main__':
    main()


In [ ]:
!python train_distillation.py

### 🤖 6.5. Train Router & Sub-Experts (Full MoE)
Once the main chat expert is distilled, train the semantic router and the sub-domain specialists.

In [ ]:
import sys
if '/content/svomptr-project' not in sys.path:
    sys.path.insert(0, '/content/svomptr-project')

from svomptr_moe.train_sub_experts import train_domain_experts
from svomptr_moe.train_router import train_router

# Set brain path to ensure weights map to Google Drive
import os
os.environ['SVOMPTR_BRAIN_PATH'] = '/content/drive/MyDrive/svomptr_brain'

print("Training Sub-Experts...")
train_domain_experts()
print("Training Router...")
train_router()


### 🚀 7. Run ML API (LocalTunnel)
Launch the FastAPI backend serving the weights from `svomptr_brain`.

In [ ]:
!npm install -g localtunnel
import subprocess
import time

print("Starting SVOMPTR FastAPI server...")
# Make sure to set SVOMPTR_BRAIN_PATH inside your repo if you want it to load weights from Drive
import os
os.environ["SVOMPTR_BRAIN_PATH"] = "/content/drive/MyDrive/svomptr_brain"

api_proc = subprocess.Popen(["uvicorn", "ml_api:app", "--host", "0.0.0.0", "--port", "8000", "--reload", "--app-dir", "/content/svomptr-project"])
time.sleep(5)

print("Starting LocalTunnel... Copy the URL below and set VITE_ML_API_URL and ML_API_URL in your Next.js/Vite frontend!")
!lt --port 8000 --subdomain svogateway9b